# ENTSO-E Transparency Platform Sandbox

In [1]:
import io
import os
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import zipfile
from datetime import datetime, timedelta, timezone

import pandas as pd

base_url = "https://web-api.tp.entsoe.eu/api"
token = os.environ["ENTSOE_TOKEN"]
period_end = datetime.now(timezone.utc).replace(hour=0, minute=0, second=0, microsecond=0)
period_start = period_end - timedelta(days=7)
params = {
    "securityToken": token,
    "documentType": "A85",
    "controlArea_Domain": "10YCH-SWISSGRIDZ",
    "periodStart": period_start.strftime("%Y%m%d%H%M"),
    "periodEnd": period_end.strftime("%Y%m%d%H%M"),
}
url = f"{base_url}?{urllib.parse.urlencode(params)}"
request = urllib.request.Request(url, headers={"Accept": "application/xml"})
with urllib.request.urlopen(request, timeout=30) as response:
    response_body = response.read()

if response_body.startswith(b"PK"):
    with zipfile.ZipFile(io.BytesIO(response_body)) as archive:
        xml_name = next(name for name in archive.namelist() if name.lower().endswith(".xml"))
        xml_response = archive.read(xml_name)
else:
    xml_response = response_body

root = ET.fromstring(xml_response)

def local_name(element):
    return element.tag.rsplit("}", 1)[-1]

def child_text(element, name):
    for child in element.iter():
        if local_name(child) == name and child.text:
            return child.text.strip()
    return None

rows = []
for time_series in (node for node in root.iter() if local_name(node) == "TimeSeries"):
    currency = child_text(time_series, "currency_Unit.name")
    price_unit = child_text(time_series, "price_Measure_Unit.name")
    unit = f"{currency}/{price_unit}" if currency and price_unit else None

    for period in (node for node in time_series if local_name(node) == "Period"):
        interval = next((node for node in period if local_name(node) == "timeInterval"), None)
        if interval is None:
            continue
        start_text = child_text(interval, "start")
        resolution_text = child_text(period, "resolution")
        if not start_text or not resolution_text:
            continue
        interval_start = pd.Timestamp(start_text)
        resolution = pd.Timedelta(resolution_text)

        for point in (node for node in period if local_name(node) == "Point"):
            position = child_text(point, "position")
            price_node = next(
                (node for node in point if local_name(node) == "imbalance_Price.amount"),
                None,
            )
            category = child_text(point, "imbalance_Price.category")
            if position and price_node is not None and price_node.text:
                point_start = interval_start + resolution * (int(position) - 1)
                rows.append({
                    "interval_start": point_start,
                    "interval_end": point_start + resolution,
                    "price_category": category,
                    "imbalance_price": float(price_node.text),
                    "unit": unit,
                })

imbalance_prices = (
    pd.DataFrame(
        rows,
        columns=["interval_start", "interval_end", "price_category", "imbalance_price", "unit"],
    )
    .sort_values(["interval_start", "price_category"])
    .reset_index(drop=True)
)
if imbalance_prices.empty:
    raise ValueError(f"No imbalance price points returned. Response begins: {xml_response[:500]!r}")

imbalance_prices["interval_start"] = pd.to_datetime(imbalance_prices["interval_start"], utc=True)
imbalance_prices["interval_end"] = pd.to_datetime(imbalance_prices["interval_end"], utc=True)
display(imbalance_prices)
print(f"{len(imbalance_prices)} points from {period_start.isoformat()} to {period_end.isoformat()}")

,interval_start,interval_end,price_category,imbalance_price,unit
0,2026-10-21 00:00:00+00:00,2026-10-21 00:15:00+00:00,A04,287.6,EUR/MWH
1,2026-10-21 00:00:00+00:00,2026-10-21 00:15:00+00:00,A05,287.6,EUR/MWH
2,2026-10-21 00:15:00+00:00,2026-10-21 00:30:00+00:00,A04,280.3,EUR/MWH
3,2026-10-21 00:15:00+00:00,2026-10-21 00:30:00+00:00,A05,280.3,EUR/MWH
4,2026-10-21 00:30:00+00:00,2026-10-21 00:45:00+00:00,A04,270.9,EUR/MWH
...,...,...,...,...,...
1319,2026-10-27 23:15:00+00:00,2026-10-27 23:30:00+00:00,A05,365.7,EUR/MWH
1320,2026-10-27 23:30:00+00:00,2026-10-27 23:45:00+00:00,A04,318.6,EUR/MWH
1321,2026-10-27 23:30:00+00:00,2026-10-27 23:45:00+00:00,A05,318.6,EUR/MWH
1322,2026-10-27 23:45:00+00:00,2026-10-28 00:00:00+00:00,A04,359.4,EUR/MWH


1324 points from 2026-10-21T00:00:00+00:00 to 2026-10-28T00:00:00+00:00
